# Runtime Breakdown Benchmark

This notebook isolates the main runtime components of the large-`N` GPU path:

- `prepare_state(...)`
- `evaluate_prepared_state(...)`
- full `compute_accelerations(...)`

Use it to compare end-to-end notebook timings against paper-style hot-path timings.

In [ ]:
import os

MANUAL_CUDA_VISIBLE_DEVICES = "2"  # set to the target GPU for this run
INDEX_PRECISION = "int32"

if MANUAL_CUDA_VISIBLE_DEVICES is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = MANUAL_CUDA_VISIBLE_DEVICES

os.environ.setdefault("JACCPOT_INDEX_PRECISION", INDEX_PRECISION)
os.environ.setdefault("YGGDRAX_INDEX_PRECISION", INDEX_PRECISION)
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("JACCPOT_INDEX_PRECISION =", os.environ.get("JACCPOT_INDEX_PRECISION"))

In [ ]:
import pathlib
import sys

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = pathlib.Path.cwd().resolve()
if not (REPO_ROOT / "jaccpot").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from jaccpot import (
    FMMAdvancedConfig,
    FMMPreset,
    FarFieldConfig,
    FastMultipoleMethod,
    NearFieldConfig,
    RuntimePolicyConfig,
    TreeConfig,
)
from yggdrax.interactions import DualTreeTraversalConfig
from examples import benchmark_utils as bench_utils

In [ ]:
runtime_particle_counts = [65536, 131072, 262144, 524288, 1048576, 2097152]
runtime_leaf_size = 128
runtime_max_order = 4
runtime_runs = 3
runtime_warmup = 1
softening = 1e-3
runtime_working_dtype = jnp.float32
runtime_key = jax.random.PRNGKey(0)

runtime_traversal_config = DualTreeTraversalConfig(
    max_pair_queue=262144,
    process_block=256,
    max_interactions_per_node=8192,
    max_neighbors_per_leaf=4096,
)

runtime_advanced = FMMAdvancedConfig(
    tree=TreeConfig(
        tree_type="radix",
        mode="lbvh",
        leaf_target=runtime_leaf_size,
        refine_local=False,
        max_refine_levels=0,
        aspect_threshold=16.0,
    ),
    farfield=FarFieldConfig(
        rotation="solidfmm",
        mode="pair_grouped",
        grouped_interactions=False,
        streamed_far_pairs=True,
        m2l_chunk_size=1024,
    ),
    nearfield=NearFieldConfig(
        mode="bucketed",
        edge_chunk_size=128,
        precompute_scatter_schedules=False,
    ),
    runtime=RuntimePolicyConfig(
        host_refine_mode="off",
        fail_fast=True,
        jit_tree=True,
        jit_traversal=True,
        memory_objective="minimum_memory",
        max_pair_queue=262144,
        pair_process_block=None,
        traversal_config=runtime_traversal_config,
        enable_interaction_cache=False,
        retain_traversal_result=False,
        retain_interactions=False,
        autotune_m2l_chunk=False,
        upward_leaf_batch_size=2048,
    ),
    mac_type="engblom",
)

runtime_fmm_kwargs = dict(
    preset=FMMPreset.LARGE_N_GPU,
    basis="solidfmm",
    theta=0.6,
    softening=softening,
    working_dtype=runtime_working_dtype,
    advanced=runtime_advanced,
)

runtime_fmm = FastMultipoleMethod(**runtime_fmm_kwargs)
print("resolved runtime memory path:", bench_utils.resolved_large_n_memory_path_report(runtime_fmm))
print("mac_type:", runtime_fmm.advanced.mac_type)
print(
    "runtime overrides @ N=2097152:",
    runtime_fmm._impl._resolve_runtime_execution_overrides(num_particles=2097152),
)

In [ ]:
def _evaluate_prepared_kwargs(fmm):
    params = inspect.signature(fmm.evaluate_prepared_state).parameters
    if "jit_traversal" in params:
        return {"jit_traversal": True}
    return {}


def benchmark_runtime_breakdown(
    particle_counts,
    *,
    fmm,
    leaf_size,
    max_order,
    runs,
    warmup,
    dtype,
    key,
):
    rows = []
    current_key = key
    eval_kwargs = _evaluate_prepared_kwargs(fmm)

    for num_particles in particle_counts:
        positions, masses, current_key = bench_utils.generate_random_distribution(
            int(num_particles),
            key=current_key,
            dtype=dtype,
        )

        prepare_timing = bench_utils.time_callable(
            fmm.prepare_state,
            positions,
            masses,
            leaf_size=int(leaf_size),
            max_order=int(max_order),
            warmup=int(warmup),
            runs=int(runs),
        )
        prepared_state = prepare_timing.result

        evaluate_timing = bench_utils.time_callable(
            fmm.evaluate_prepared_state,
            prepared_state,
            warmup=int(warmup),
            runs=int(runs),
            **eval_kwargs,
        )

        full_timing = bench_utils.time_callable(
            fmm.compute_accelerations,
            positions,
            masses,
            leaf_size=int(leaf_size),
            max_order=int(max_order),
            reuse_prepared_state=False,
            warmup=int(warmup),
            runs=int(runs),
        )

        rows.append(
            {
                "num_particles": int(num_particles),
                "prepare_mean_s": float(prepare_timing.mean),
                "prepare_std_s": float(prepare_timing.std),
                "evaluate_mean_s": float(evaluate_timing.mean),
                "evaluate_std_s": float(evaluate_timing.std),
                "full_mean_s": float(full_timing.mean),
                "full_std_s": float(full_timing.std),
                "prepare_plus_eval_mean_s": float(prepare_timing.mean + evaluate_timing.mean),
                "prepare_over_full": float(prepare_timing.mean / max(full_timing.mean, 1e-12)),
                "evaluate_over_full": float(evaluate_timing.mean / max(full_timing.mean, 1e-12)),
            }
        )

    return pd.DataFrame(rows).sort_values("num_particles").reset_index(drop=True)

In [ ]:
import inspect

runtime_breakdown_df = benchmark_runtime_breakdown(
    runtime_particle_counts,
    fmm=runtime_fmm,
    leaf_size=runtime_leaf_size,
    max_order=runtime_max_order,
    runs=runtime_runs,
    warmup=runtime_warmup,
    dtype=runtime_working_dtype,
    key=runtime_key,
)
runtime_breakdown_df

In [ ]:
plot_df = runtime_breakdown_df.copy()
x = plot_df["num_particles"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(x, plot_df["prepare_mean_s"], marker="o", label="prepare_state")
axes[0].plot(x, plot_df["evaluate_mean_s"], marker="o", label="evaluate_prepared_state")
axes[0].plot(x, plot_df["full_mean_s"], marker="o", label="compute_accelerations")
axes[0].set_xscale("log", base=2)
axes[0].set_yscale("log")
axes[0].set_xlabel("Number of particles")
axes[0].set_ylabel("Mean runtime [s]")
axes[0].set_title("Runtime Breakdown")
axes[0].legend()

axes[1].plot(x, plot_df["prepare_over_full"], marker="o", label="prepare / full")
axes[1].plot(x, plot_df["evaluate_over_full"], marker="o", label="evaluate / full")
axes[1].axhline(1.0, color="black", linewidth=1, linestyle="--")
axes[1].set_xscale("log", base=2)
axes[1].set_xlabel("Number of particles")
axes[1].set_ylabel("Fraction of full runtime")
axes[1].set_title("Relative Cost")
axes[1].legend()

plt.tight_layout()

## Interpretation

Use this notebook to compare:

- hot prepared evaluation (`evaluate_prepared_state`)
- one-shot setup (`prepare_state`)
- end-to-end cost (`compute_accelerations`)

If a paper reports very small runtimes, check whether it is closer to the prepared evaluation column than to the full end-to-end column.